# 05 — Identifiability checks S15 (E5)
Results §7, S15. **LOAD+VERIFY** (v6 PCA redteam).

**Provenance.** Built by `~/.claude/skills/repro-notebook/SKILL.md` (Phase 3).
Companion docs: `docs/PAPER/repro/{MANIFEST.md, MAP.md, REPORT.md}`.
Target paper: `docs/PAPER/main.tex` → `Results/results_v4.tex`.
Helpers: `docs/PAPER/repro/_repro_util.py`.

**Modes.** `RECOMPUTE` = computed locally from C010 amplitudes. `LOAD+VERIFY` =
read the committed result JSON and compare to the printed paper value. SRM/BrainIAK
numbers are read from committed JSON (no MPI in this kernel).

**Source & code map** (`future_phase2_filter_optimization/results/redteam/`)

| id | reported | source JSON | mode |
|---|---|---|---|
| E5.2 | f10 0.26 / 0.14 | `param_recovery_voxel_v6_pca_v2.json` (mean of per-donor `frac_within_10deg`) | LOAD+VERIFY |
| E5.4 | HC pseudo-CVD rank 0.875 | `verdict_matrix_v6_pca_v2.json` `specificity.rank_distance` | LOAD+VERIFY |
| E5.5 | label-perm p 0.167 / 0.471 | `verdict_matrix_v6_pca_v2.json` `within_subject_sig.p_perm` | LOAD+VERIFY |

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))   # docs/PAPER/repro
import numpy as np
import _repro_util as U
U._RESULTS.clear()   # fresh check log per notebook
print("repo:", U.REPO)

repo: /Users/jinilkim/Library/CloudStorage/OneDrive-Personal/Projects/colorBlind_analysis


### E5.2 — voxel parameter recovery f₁₀° (aggregate over 7 HC donors)

In [2]:
j = U.load_json(U.P2 / "results/redteam/param_recovery_voxel_v6_pca_v2.json")
cells = j["cells"]
def f10(cand):
    vals = [c["frac_within_10deg"] for k, c in cells.items() if k.startswith(cand) and "frac_within_10deg" in c]
    return float(np.mean(vals)) if vals else float("nan")
U.check("E5.2 S08-robust f10", f10("S08-robust"), 0.26, tol=0.03)
U.check("E5.2 S09-primary f10", f10("S09-primary"), 0.14, tol=0.03)

[OK ] E5.2 S08-robust f10: produced=0.2642857142857143  reported=0.26
[OK ] E5.2 S09-primary f10: produced=0.13571428571428573  reported=0.14


### E5.4 — HC pseudo-CVD specificity rank (Test 2b)
From the aggregated `verdict_matrix_v6_pca_v2.json` (`specificity.rank_distance`).

In [3]:
vm = U.load_json(U.P2 / "results/redteam/verdict_matrix_v6_pca_v2.json")["per_candidate"]
for cand in ["S08-robust", "S09-primary"]:
    U.check(f"E5.4 {cand} rank_distance", vm[cand]["specificity"]["rank_distance"], 0.875, tol=0.001)

[OK ] E5.4 S08-robust rank_distance: produced=0.875  reported=0.875
[OK ] E5.4 S09-primary rank_distance: produced=0.875  reported=0.875


### E5.5 — color-label permutation p (Test 2c)
From `verdict_matrix` (`within_subject_sig.p_perm`, `real_loss`).

In [4]:
for cand, rep in [("S08-robust", 0.167), ("S09-primary", 0.471)]:
    ws = vm[cand]["within_subject_sig"]
    U.check(f"E5.5 {cand} p_perm", ws["p_perm"], rep, tol=0.005)
    print(f"     {cand}: real_loss={ws['real_loss']:.3f}  perm_5pct={ws['perm_loss_5pct']:.3f}")

[OK ] E5.5 S08-robust p_perm: produced=0.16683316683316685  reported=0.167
     S08-robust: real_loss=-2.892  perm_5pct=-3.136
[OK ] E5.5 S09-primary p_perm: produced=0.47052947052947053  reported=0.471
     S09-primary: real_loss=-1.681  perm_5pct=-3.053


In [5]:
U.summary()


=== 6/6 checks reproduced ===
